# Tarea 1. Compensación de iluminación y detección de personas

**Procesamiento de Imágenes — Universidad Diego Portales**

Las imágenes de la base DARK FACE son escenas nocturnas reales: casi toda la información
vive en un puñado de niveles bajos. Sobre ellas se aplican distintas técnicas de mejora de
contraste y se mide el efecto con un detector de personas (YOLO), que es una de las formas de
evaluar el realce cuando no existe una imagen de referencia con la exposición correcta.

Cada método clásico se ejecuta en dos variantes: aplicando la transformación a **cada canal
BGR por separado** (`bgr=True`) o solo sobre el **canal de luminancia** (`bgr=False`). Parte
del trabajo es decidir cuál conviene, y con qué criterio se decide eso.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Preparación del entorno

In [1]:
# En Colab / entorno limpio, comentar una vez instalado y clonado el repo.
!pip install -q ultralytics
!git clone -q https://github.com/Li-Chongyi/Zero-DCE.git

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch

# Puede activar el uso de la GPU para hacer inferencias rápidas de las redes neuronales.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", DEVICE)

# --- rutas de las bases de datos, dejo mi ruta como ejemplo. Dentro de la ruta está la carpeta people.
BASE        = "/content/drive/MyDrive/Procesamiento de Imágenes/clases listas/clase 5"
PATH_PEOPLE = os.path.join(BASE, "people")                   # DARK FACE (sin referencia)
PATH_ZDCE   = os.path.join(BASE, "Zero-DCE/Zero-DCE_code")   # repositorio de Zero-DCE

# --- parámetros de la tarea -------------------------------------------------
GAMMA     = 0.4      # exponente de la transformación de potencia
PCT_LO    = 5        # percentil inferior del estiramiento
PCT_HI    = 95       # percentil superior del estiramiento
CONF_YOLO = 0.5      # confianza mínima para aceptar una detección

fatal: destination path 'Zero-DCE' already exists and is not an empty directory.
dispositivo: cpu


## 1. Funciones auxiliares

In [5]:
# Importe aqui las funciones auxiliares escritas en un archivo llamado funciones.py dentro de una carpeta llamada utils.
from funciones import (
    histograma,
    triptico,
    mostrar,
    niveles_usados,
    entropia,
    mostrar_histogramas,
)

## 2. Zero-DCE

Zero-DCE no predice la imagen realzada: predice **curvas de ajuste** que se aplican de
forma iterativa a cada píxel. Es decir, aprende LUTs, pero una por píxel y por iteración,
en vez de una sola para toda la imagen. Son unos 79 mil parámetros.

Ojo con el orden de canales: la red espera **RGB** normalizado a [0, 1], no BGR.

In [6]:
import importlib.util, glob

# Busca model.py: primero en PATH_ZDCE, si no, donde haya quedado el clon
candidatos = ([os.path.join(PATH_ZDCE, "model.py")] +
              glob.glob("**/Zero-DCE_code/model.py", recursive=True) +
              glob.glob("/content/**/Zero-DCE_code/model.py", recursive=True))
candidatos = [c for c in candidatos if os.path.isfile(c)]
assert candidatos, "no encuentro model.py; revisa dónde quedó el clon de Zero-DCE"

PATH_ZDCE  = os.path.abspath(os.path.dirname(candidatos[0]))
PESOS_ZDCE = os.path.join(PATH_ZDCE, "snapshots", "Epoch99.pth")
assert os.path.isfile(PESOS_ZDCE), f"faltan los pesos en {PESOS_ZDCE}"
print("usando:", PATH_ZDCE)

_spec = importlib.util.spec_from_file_location("zdce_model", os.path.join(PATH_ZDCE, "model.py"))
zdce_model = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(zdce_model)

_dce_net = None

def cargar_zero_dce(pesos=None):
    """Carga los pesos preentrenados una sola vez."""
    global _dce_net
    if _dce_net is None:
        net = zdce_model.enhance_net_nopool()
        estado = torch.load(pesos or PESOS_ZDCE, map_location="cpu")
        estado = {k.replace("module.", ""): v for k, v in estado.items()}
        net.load_state_dict(estado)
        _dce_net = net.eval().to(DEVICE)
    return _dce_net


@torch.no_grad()
def zero_dce(img, bgr=None):
    """Realce con Zero-DCE. Entra y sale BGR uint8.

    El argumento bgr se ignora: la red procesa la imagen completa y no admite la
    distinción entre 'por canal' y 'solo luminancia'.
    """
    net = cargar_zero_dce()
    rgb = cv2.cvtColor(np.ascontiguousarray(img), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    x = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    _, realzada, _ = net(x)
    out = realzada.squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()
    out = cv2.cvtColor((out * 255).round().astype(np.uint8), cv2.COLOR_RGB2BGR)
    return out

print("parámetros de la red:", sum(p.numel() for p in cargar_zero_dce().parameters()))

usando: /Users/benjaminzunigapueller/Desktop/ProcesaientoDeImagenes/Tareas/Tarea1/Zero-DCE/Zero-DCE_code
parámetros de la red: 79416


## 3. Detector de personas

YOLOv8 con `classes=[0]` (clase *person* de COCO) y confianza mínima 0.5. Se registran el
número de detecciones **y la confianza media**: el conteo es un entero pequeño que salta de
forma brusca, mientras que la confianza media se mueve de manera mucho más suave.

`cv2.imread` entrega BGR y Ultralytics asume BGR cuando recibe un arreglo de NumPy, así que
la imagen se pasa directamente, sin invertir canales.

In [7]:
from ultralytics import YOLO

modelo_yolo = YOLO("yolov8n.pt")     # se descarga solo la primera vez


def detectar_personas(img, conf=CONF_YOLO, dibujar=True):
    """Detecta personas. Devuelve (n_detecciones, confianza_media, imagen_anotada)."""
    res = modelo_yolo(img, classes=[0], conf=conf, verbose=False)[0]
    confianzas = res.boxes.conf.cpu().numpy()

    anotada = None
    if dibujar:
        anotada = img.copy()
        cajas = res.boxes.xyxy.cpu().numpy().astype(int)
        for (x1, y1, x2, y2), c in zip(cajas, confianzas):
            cv2.rectangle(anotada, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(anotada, f"{c:.2f}", (x1, max(y1 - 6, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

    media = float(confianzas.mean()) if len(confianzas) else 0.0
    return len(confianzas), media, anotada

### A.1 Inspección de una imagen

Antes de correr todo, elige una una imagen, despliégala y revisa su histograma. Anota en qué rango de niveles vive
la información y qué fracción de píxeles está pegada a los extremos.

In [10]:
img = cv2.imread("people/10.png")
histograma(img, "histograma original")

TypeError: histograma() takes 1 positional argument but 2 were given

In [ ]:
I'd be happy to help fix the error, but I don't see an error message in your message. Could you please:

1. **Share the error message** you're getting (the full traceback)
2. **Specify which cell** is causing the problem
3. **Describe what you're trying to do** in that cell

Once you provide the error details, I can give you a targeted fix.

### A.2 Efecto de cada método sobre la imagen de ejemplo

Para la imagen escogida, compara las dos variantes de cada método. Fíjate especialmente en la dominante de color de
la escena y en qué le pasa con cada variante. Inclute Zero-DCE

### A.3 Visaliza la función de transformación.

Usando la imagen escogida, visualiza la imagen resultante, la función de transformación y el histograma del canal Y de YCbCr. Comenta cada resultado. Usa la función tríptico vista en clase, adaptando lo necesario.

### A.4 Detección sobre todas las imágenes

Se recorre la base completa, se aplica cada método en sus dos variantes y se detectan
personas. Las imágenes anotadas se guardan en `resultados/parte_a/` como parte de los entregables. Muestre la tabla en formato de pandas

### A.5 Detección sobre todas las imágenes

Construya una segunda tabla con la misma estructura, pero registrando la confianza media de las detecciones en lugar del conteo (0 si no hubo ninguna detección). La última columna corresponde al promedio de las confianzas. Muestre los resultados en orden descendente. Muestre la tabla en formato de pandas

### A.6 Sensibilidad al umbral

Repita la tabla 1 con una confianza mínima de 0.25 y compare el orden de los métodos con el obtenido a 0.5. Analice si hay consistencia entre los métodos variando el umbral.  